In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('nhanes_data/nhanes_cleaned.csv')
print(f'Loaded: {df.shape}')

# ─────────────────────────────────────────────────────────
# FEATURE 1: Estimated glycemic load
# We don't have per-food GI in NHANES, so we estimate:
# GL_estimated = (carb_avg_g * weighted_avg_GI) / 100
# Weighted avg GI approximation:
#   sugars contribute GI ~65, non-sugar carbs average ~55
# ─────────────────────────────────────────────────────────
df['sugar_carb_ratio'] = (df['sugars_avg'] / df['carb_avg'].clip(lower=1)).clip(0,1)
df['estimated_gi'] = df['sugar_carb_ratio'] * 65 + (1 - df['sugar_carb_ratio']) * 55
df['feature_glycemic_load'] = (df['carb_avg'] * df['estimated_gi'] / 100).clip(lower=0)

# ─────────────────────────────────────────────────────────
# FEATURE 2: Refined carbohydrate share
# Proxy: (carbs - fiber*2) / carbs  (fiber correlates with whole foods)
# Clipped to 0-1
# ─────────────────────────────────────────────────────────
df['feature_refined_carb_share'] = (
    (df['carb_avg'] - df['fiber_avg'].fillna(0) * 2) / df['carb_avg'].clip(lower=1)
).clip(0, 1)

# ─────────────────────────────────────────────────────────
# FEATURE 3: Dietary fiber per 1000 kcal (normalised for energy)
# ─────────────────────────────────────────────────────────
df['feature_fiber_per_1000kcal'] = (
    df['fiber_avg'] / df['energy_avg'].clip(lower=500) * 1000
).clip(lower=0)

# ─────────────────────────────────────────────────────────
# FEATURE 4: Quality protein share
# NHANES does not break protein by source, so we use
# total protein % of energy as a proxy for protein adequacy
# ─────────────────────────────────────────────────────────
df['feature_protein_pct_energy'] = (
    df['protein_avg'] * 4 / df['energy_avg'].clip(lower=500)
).clip(0, 0.5)

# ─────────────────────────────────────────────────────────
# FEATURE 5: Saturated fat % of energy
# ─────────────────────────────────────────────────────────
df['feature_sfa_pct_energy'] = (
    df['sat_fat_avg'] * 9 / df['energy_avg'].clip(lower=500)
).clip(0, 0.4)

# ─────────────────────────────────────────────────────────
# FEATURE 6: MUFA:SFA ratio
# Direct from NHANES — this is one advantage over rule-based
# ─────────────────────────────────────────────────────────
df['feature_mufa_sfa_ratio'] = (
    df['mufa_avg'] / df['sat_fat_avg'].clip(lower=0.1)
).clip(0, 6)

# ─────────────────────────────────────────────────────────
# FEATURE 7: Sodium mg per day
# ─────────────────────────────────────────────────────────
df['feature_sodium_mg'] = df['sodium_avg'].clip(lower=0, upper=8000)

# ─────────────────────────────────────────────────────────
# Collect features and drop rows with missing values
# ─────────────────────────────────────────────────────────
FEATURE_COLS = [
    'feature_glycemic_load',
    'feature_refined_carb_share',
    'feature_fiber_per_1000kcal',
    'feature_protein_pct_energy',
    'feature_sfa_pct_energy',
    'feature_mufa_sfa_ratio',
    'feature_sodium_mg',
]

TARGET_COLS = ['outcome_diabetes', 'outcome_cvd']
META_COLS   = ['participant_id', 'dietary_weight', 'race_ethnicity',
               'age_years', 'gender', 'bmi', 'cycle']

model_df = df[FEATURE_COLS + TARGET_COLS + META_COLS].dropna()
print(f'Model dataset after dropping NaN: {model_df.shape}')
print(f'\nFeature summary:')
print(model_df[FEATURE_COLS].describe().round(2))

model_df.to_csv('nhanes_data/nhanes_features.csv', index=False)
print('\nSaved: nhanes_data/nhanes_features.csv')


Loaded: (19509, 26)
Model dataset after dropping NaN: (18835, 16)

Feature summary:
       feature_glycemic_load  feature_refined_carb_share  \
count               18835.00                    18835.00   
mean                  145.43                        0.85   
std                    64.84                        0.07   
min                     8.62                        0.09   
25%                   100.73                        0.82   
50%                   135.24                        0.87   
75%                   178.03                        0.90   
max                   894.68                        1.00   

       feature_fiber_per_1000kcal  feature_protein_pct_energy  \
count                    18835.00                    18835.00   
mean                         8.70                        0.16   
std                          4.17                        0.05   
min                          0.00                        0.00   
25%                          5.81                 